# Insurance Charges Regression

Predict healthcare insurance charges from demographic and lifestyle features using linear regression in scikit-learn.

**Cross-validated R²:** 0.7451


In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# 1. Load data
df = pd.read_csv("insurance.csv")

# 2. Clean data
df = df.dropna()

df["sex"] = (
    df["sex"].astype(str).str.strip().str.lower()
    .replace({"m": "male", "man": "male", "f": "female", "woman": "female"})
)
df["smoker"] = df["smoker"].astype(str).str.strip().str.lower()
df["region"] = df["region"].astype(str).str.strip().str.lower()
df["charges"] = (
    df["charges"].astype(str)
    .replace(r"[\$,]", "", regex=True)
    .astype(float)
)
df = df[df["age"] > 0]
df.loc[df["children"] < 0, "children"] = 0

# 3. Features and target
X = df.drop("charges", axis=1)
y = df["charges"]

# 4. Encode categorical features
X = pd.get_dummies(
    X,
    columns=["sex", "smoker", "region"],
    drop_first=True
)

# 5. Scale numerical features
numeric_columns = ["age", "bmi", "children"]
scaler = StandardScaler()
X[numeric_columns] = scaler.fit_transform(X[numeric_columns])

# 6. Create and evaluate model
insurance_model = LinearRegression()
scores = cross_val_score(
    insurance_model,
    X,
    y,
    cv=5,
    scoring="r2"
)
r2_score = scores.mean()
print("R2 Score:", r2_score)

# 7. Train on all available training data
insurance_model.fit(X, y)

# 8. Load and clean validation data
validation_data = pd.read_csv("validation_dataset.csv")

validation_data["sex"] = (
    validation_data["sex"].astype(str).str.strip().str.lower()
    .replace({"m": "male", "man": "male", "f": "female", "woman": "female"})
)
validation_data["smoker"] = validation_data["smoker"].astype(str).str.strip().str.lower()
validation_data["region"] = validation_data["region"].astype(str).str.strip().str.lower()
validation_data.loc[validation_data["children"] < 0, "children"] = 0

# 9. Prepare validation features
validation_features = pd.get_dummies(
    validation_data,
    columns=["sex", "smoker", "region"],
    drop_first=True
)
validation_features = validation_features.reindex(columns=X.columns, fill_value=0)
validation_features[numeric_columns] = scaler.transform(validation_features[numeric_columns])

# 10. Predict charges and apply $1,000 minimum
predicted_charges = insurance_model.predict(validation_features)
predicted_charges = np.maximum(predicted_charges, 1000)
validation_data["predicted_charges"] = predicted_charges

validation_data.head()
